# 05 — Volume-weighted delays per station

**ENAI 603 Capstone — WMATA Metro Delay Prediction**

This notebook provides an interactive visualization of volume of delays vs delay rate per station (WMATA Rail System)

**Sections:**
1. Load features.csv and WMATA Station dataset files
2. Plot volume of delays vs delay rate per station
3. Top 10 stations with worst volume vs delay rate ratio

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px

PROJECT_DIR = Path("..")
DB_PATH = PROJECT_DIR / "data" / "wmata.db"

conn = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH}")
print(f"Database size: {DB_PATH.stat().st_size / 1024:.0f} KB")

## 1. Load features.csv and WMATA Station dataset files

In [ ]:
# Load features.csv file
df = pd.read_csv("../data/features.csv")
df_features = df[df['is_orphan'] == 0]
df_features.head(5)

In [ ]:
# Load stations dataset
df_stations = pd.read_sql("SELECT * FROM stations", conn)
print(f"Total stations: {len(df_stations)}")
df_stations.head(5)

## 2. Plot volume of delays vs delay rate per station

In [ ]:
# Add volume of delays per station
volume_vs_delay_df = df_features.groupby(['location_code', 'location_name']).agg(arrivals=('is_delayed', 'size'), delays=('is_delayed', 'sum')).reset_index()

# Add code of lines that serve each station
volume_vs_delay_df = pd.merge(volume_vs_delay_df, df_stations[['station_code', 'line_code1', 'line_code2', 'line_code3', 'line_code4']], left_on='location_code', right_on='station_code', how='left').drop(columns=['station_code'])

# Add delay rate per station
volume_vs_delay_df['delay_rate'] = (volume_vs_delay_df['delays'] / volume_vs_delay_df['arrivals']).round(2)

volume_vs_delay_df.head()

In [ ]:
# Color stations by primary line code
color_map = {'RD': 'red',
             'BL': 'blue',
             'YL': 'yellow',
             'OR': 'orange',
             'GR': 'green',
             'SV': 'grey'}

fig = px.scatter(volume_vs_delay_df, x="delays", y="delay_rate", title='Volume of delays vs Delay rate per station', color='line_code1', color_discrete_map=color_map,
                 hover_name="location_name", hover_data=["delays", "delay_rate"]).update_layout(
                 xaxis_title="Volume of delays", yaxis_title="Delay Rate")
fig.show()

In [ ]:
# Save Plotly interactive chart as a standalone HTML file
fig.write_html("volume_vs_delays_per_station.html")

## 3. Top 10 stations with worst volume vs delay rate ratio

In [ ]:
top_10 = volume_vs_delay_df.sort_values(['delay_rate', 'delays'], ascending=[False, False]).head(11).reset_index(drop=True).reindex(range(1, 11))
top_10[['location_code', 'location_name', 'line_code1', 'arrivals', 'delays', 'delay_rate']]